In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

os.environ['TF_CUDNN_USE_AUTOTUNE'] = '0'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import tensorflow as tf
from keras import layers, models, losses, regularizers
from keras.models import load_model
from keras.callbacks import ModelCheckpoint
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

print("GPU Trovate:", len(tf.config.list_physical_devices('GPU')))
for gpu in tf.config.list_physical_devices('GPU'):
    print("Nome:", gpu.name)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memoria GPU configurata in modalità dinamica.")
    except RuntimeError as e:
        print("Errore GPU:", e)

tf.keras.backend.clear_session()

physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    try:
        for gpu in physical_devices:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ GPU Memory Growth abilitato con successo.")
    except RuntimeError as e:
        print(f"Errore configurazione GPU: {e}")

GPU Trovate: 1
Nome: /physical_device:GPU:0
Memoria GPU configurata in modalità dinamica.
✅ GPU Memory Growth abilitato con successo.


In [ ]:
def embedded_summary(model, input_shape=(1, 120, 18), is_int8=False):
    total_params = model.count_params()
    
    bytes_per_param = 1 if is_int8 else 4
    estimated_flash_kb = (total_params * bytes_per_param) / 1024
    
    bytes_per_activation = 1 if is_int8 else 4
    max_adjacent_ram_kb = 0
    
    previous_layer_size = (np.prod(input_shape) * bytes_per_activation) / 1024
    
    for layer in model.layers:
        if layer.__class__.__name__ == 'InputLayer' or not hasattr(layer, 'output_shape'):
            continue
            
        output_shape = layer.output_shape
        if isinstance(output_shape, list):
            num_elements = sum([np.prod([dim for dim in shape[1:] if dim is not None]) for shape in output_shape])
        else:
            num_elements = np.prod([dim for dim in output_shape[1:] if dim is not None])
            
        current_layer_size = (num_elements * bytes_per_activation) / 1024
        
        current_peak = previous_layer_size + current_layer_size
        
        if current_peak > max_adjacent_ram_kb:
            max_adjacent_ram_kb = current_peak
            
        previous_layer_size = current_layer_size

    print("============================================")
    mode_str = "INT8 (Quantizzato)" if is_int8 else "FLOAT32 (Training)"
    print(f"   REPORT REQUISITI ESP32-S3 [{mode_str}]   ")
    print("============================================")
    print(f" Memoria FLASH stimata : ~{estimated_flash_kb:.2f} KB  (Limite: 800 KB)")
    print(f" Memoria SRAM stimata  : ~{max_adjacent_ram_kb:.2f} KB (Limite: 300 KB)")
    print("============================================\n")

In [ ]:
# HUNGARIAN LOSS 

import itertools
PERM_INDICES = tf.constant(list(itertools.permutations([0, 1, 2, 3])), dtype=tf.int32)
ROOM_DIMS = tf.constant([4.8, 7.2], dtype=tf.float32)

def hungarian_total_loss(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1) 
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)

    y_true_coords_exp = tf.expand_dims(y_true_coords, 1) 
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3]) 
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)

    y_pred_mask_perm_safe = tf.clip_by_value(y_pred_mask_perm, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm_safe)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3]) 

    total_cost = coords_cost_norm + (1.5 * mask_cost_norm) 
    return tf.reduce_min(total_cost, axis=1) 

def hungarian_rmse_metres(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    
    min_coords_cost = tf.reduce_min(coords_cost, axis=1)
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    
    return tf.sqrt(min_coords_cost / num_valid_people)

def hungarian_mask_acc(y_true, y_pred):
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1)) 
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1)) 
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)
    
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)
    
    #bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    y_pred_mask_perm_safe = tf.clip_by_value(y_pred_mask_perm, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm_safe)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3])
    
    total_cost = coords_cost_norm + (1.5 * mask_cost_norm)
    best_perm_idx = tf.argmin(total_cost, axis=1, output_type=tf.int32)
    
    batch_size = tf.shape(y_pred)[0]
    gather_nd_indices = tf.stack([tf.range(batch_size, dtype=tf.int32), best_perm_idx], axis=1)
    best_mask_pred = tf.gather_nd(y_pred_mask_perm, gather_nd_indices)
    
    return tf.reduce_mean(tf.keras.metrics.binary_accuracy(y_true_mask, best_mask_pred))

2026-07-14 10:36:24.875633: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-07-14 10:36:24.876070: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-07-14 10:36:24.876086: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
I0000 00:00:1784018184.876678 4876871 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1784018184.877097 4876871 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [ ]:
# DATA ENGINE (Normalizzazione [0, 1]) best

def load_and_process_all_files(file_list, alpha=0.02):
    X_all, Y_all = [], []
    print(f"Inizio caricamento ed EMA Decluttering di {len(file_list)} file...")
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   
        people_xy = data['people_xy'].astype(np.float32) 
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
        mag_reshaped = mag.reshape(T, 1, 120, 18) 
        
        bg = np.copy(mag_reshaped[0])
        decluttered = np.zeros_like(mag_reshaped)
        
        for t in range(T):
            bg = alpha * mag_reshaped[t] + (1 - alpha) * bg
            decluttered[t] = np.abs(mag_reshaped[t] - bg)
        
        flat_coords = people_xy.reshape(T, 8)
        combined_target = np.concatenate([flat_coords, people_mask], axis=1)

        X_all.append(decluttered)
        Y_all.append(combined_target)
        
        print(f"File {i+1}/{len(file_list)} processato.")

    X = np.concatenate(X_all, axis=0).astype(np.float32)
    Y = np.concatenate(Y_all, axis=0).astype(np.float32)
    return X, Y



# Split1 
#val_indices = [23, 20, 3, 15, 7, 11] 
#train_indices = [22, 16, 17, 18, 19, 21, 0, 1, 2, 4, 12, 13, 14, 5, 6, 8, 9, 10]
# Split2 
val_indices = [23, 20, 0, 13, 9] 
train_indices = [22, 16, 17, 18, 19, 21, 1, 2, 3, 4, 12, 14, 15, 5, 6, 7, 8, 10, 11]
# Split3 
#val_indices = [22, 20, 2, 12, 14, 6] 
#train_indices = [23, 16, 18, 19, 21, 0, 1, 3, 4, 13, 15, 5, 7, 8, 9, 10, 11, 17]

tutti_i_file = glob.glob("dataset/data/*.npz")
#tutti_i_file = glob.glob("dataset/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]


print("\n--- PREPARAZIONE TRAINING SET ---")
X_train_raw, Y_train = load_and_process_all_files(train_files)

print("\n--- PREPARAZIONE VALIDATION SET ---")
X_val_raw, Y_val = load_and_process_all_files(val_files)


GLOBAL_MAX = np.percentile(X_train_raw, 99.5)
print(f"\n---> GLOBAL_MAX CALCOLATO: {GLOBAL_MAX:.2f} <---")
print("INSERISCI QUESTO VALORE NEL TUO CODE.PY PER L'INFERENZA!")

X_train = np.clip(X_train_raw, 0, GLOBAL_MAX) / GLOBAL_MAX
X_val = np.clip(X_val_raw, 0, GLOBAL_MAX) / GLOBAL_MAX

print("\n==================================================")
print(f"DATI TOTALI PRONTI E NORMALIZZATI IN RAM!")
print(f"Totale FRAME individuali di Train:      {X_train.shape[0]}")
print(f"Totale FRAME individuali di Validation: {X_val.shape[0]}")
print("==================================================")

In [ ]:
# DATA ENGINE (Normalizzazione [0, 1] + Data Augmentation)
def load_and_process_all_files(file_list, alpha=0.02):
    X_all, Y_all = [], []
    print(f"Inizio caricamento ed EMA Decluttering di {len(file_list)} file...")
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   
        people_xy = data['people_xy'].astype(np.float32) 
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
        mag_reshaped = mag.reshape(T, 1, 120, 18) 
        
        bg = np.copy(mag_reshaped[0])
        decluttered = np.zeros_like(mag_reshaped)
        
        for t in range(T):
            bg = alpha * mag_reshaped[t] + (1 - alpha) * bg
            decluttered[t] = np.abs(mag_reshaped[t] - bg)
        
        flat_coords = people_xy.reshape(T, 8)
        combined_target = np.concatenate([flat_coords, people_mask], axis=1)

        X_all.append(decluttered)
        Y_all.append(combined_target)
        
        print(f"File {i+1}/{len(file_list)} processato.")

    X = np.concatenate(X_all, axis=0).astype(np.float32)
    Y = np.concatenate(Y_all, axis=0).astype(np.float32)
    return X, Y



# Split1 (78% train e 22% val), > windows con 3/4 persone in train
val_indices = [23, 20, 3, 15, 7, 11] 
train_indices = [22, 16, 17, 18, 19, 21, 0, 1, 2, 4, 12, 13, 14, 5, 6, 8, 9, 10]
# Split2 
#val_indices = [23, 20, 0, 13, 9] 
#train_indices = [22, 16, 17, 18, 19, 21, 1, 2, 3, 4, 12, 14, 15, 5, 6, 7, 8, 10, 11]
# Split3 
#val_indices = [22, 20, 2, 12, 14, 6] 
#train_indices = [23, 16, 18, 19, 21, 0, 1, 3, 4, 13, 15, 5, 7, 8, 9, 10, 11, 17]

tutti_i_file = glob.glob("dataset/data/*.npz")
#tutti_i_file = glob.glob("dataset/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]


print("\n--- PREPARAZIONE TRAINING SET ---")
X_train_raw, Y_train = load_and_process_all_files(train_files)

print("\n--- PREPARAZIONE VALIDATION SET ---")
X_val_raw, Y_val = load_and_process_all_files(val_files)


GLOBAL_MAX = np.percentile(X_train_raw, 99.5)
print(f"\n---> GLOBAL_MAX CALCOLATO: {GLOBAL_MAX:.2f} <---")
print("INSERISCI QUESTO VALORE NEL TUO CODE.PY PER L'INFERENZA!")

X_train = np.clip(X_train_raw, 0, GLOBAL_MAX) / GLOBAL_MAX
X_val = np.clip(X_val_raw, 0, GLOBAL_MAX) / GLOBAL_MAX

print("\n==================================================")
print(f"DATI TOTALI PRONTI E NORMALIZZATI IN RAM!")
print(f"Totale FRAME individuali di Train:      {X_train.shape[0]}")
print(f"Totale FRAME individuali di Validation: {X_val.shape[0]}")
print("==================================================")


# RADAR DROPOUT

def radar_dropout_augmentation(x, y):
    drop_rate = 0.20 # 20% di probabilità di spegnere un intero radar
    x_reshaped = tf.reshape(x, (1, 120, 6, 3))
    random_tensor = tf.random.uniform(shape=(1, 1, 6, 1))
    keep_mask = tf.cast(random_tensor >= drop_rate, tf.float32)
    x_augmented = (x_reshaped * keep_mask) / (1.0 - drop_rate)
    x_final = tf.reshape(x_augmented, (1, 120, 18))
    return x_final, y

print("\n--- PREPARAZIONE TF.DATA PIPELINE CON RADAR DROPOUT ---")

with tf.device('/CPU:0'):
    # Il Train Dataset riceve Shuffle, Radar Dropout (tramite .map) e Batching
    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, Y_train))
    train_dataset = train_dataset.shuffle(buffer_size=5000)
    train_dataset = train_dataset.map(radar_dropout_augmentation, num_parallel_calls=tf.data.AUTOTUNE)
    train_dataset = train_dataset.batch(32).prefetch(tf.data.AUTOTUNE)
    
    # Il Validation Dataset NON riceve il dropout
    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, Y_val))
    val_dataset = val_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

In [ ]:
# ARCHITETTURA EEAI-NET V1 


def squeeze_excite_block_2d(x, filters, r=8):
    """Meccanismo di Attenzione spaziale basato su Squeeze-and-Excitation"""
    # Squeeze
    se = layers.GlobalAveragePooling2D()(x)
    # Excitation
    se = layers.Dense(max(1, filters // r), activation='relu', use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    # Reshape 
    se = layers.Reshape((1, 1, filters))(se)
    return layers.Multiply()([x, se])

def residual_reduction_module_2d(x, filters, r=8, name_prefix=""):
    """
    RRM: Red(Res(x)) 
    Unisce una skip connection pesata dal SE block e un dimezzamento 
    parallelo della dimensione temporale (range bins).
    """
    # Residual Branch 
    res = layers.Conv2D(filters, kernel_size=(1, 3), padding='same', activation='relu', 
                        name=f"{name_prefix}_res_conv")(x)
    res = squeeze_excite_block_2d(res, filters, r=r)
    res = layers.Add(name=f"{name_prefix}_res_add")([res, x]) # Skip connection

    # Reduction Branch
    red1 = layers.Conv2D(filters, kernel_size=(1, 3), strides=(1, 2), padding='same', 
                         activation='relu', name=f"{name_prefix}_red_conv1")(res)
    red2 = layers.Conv2D(filters, kernel_size=(1, 1), strides=(1, 2), padding='same', 
                         activation='relu', name=f"{name_prefix}_red_conv2")(res)
    
    out = layers.Add(name=f"{name_prefix}_red_add")([red1, red2])
    return out

def build_eeai_model_v2_rrm(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")
    
    F = 64 
    r = 8  # Reduction ratio 
    
    # Feature Extraction
    x = layers.Conv2D(F, kernel_size=(1, 5), padding='same', activation='relu', name="init_conv")(inputs)
    
    
    x = residual_reduction_module_2d(x, filters=F, r=r, name_prefix="rrm1") # Bins: 120 -> 60
    x = residual_reduction_module_2d(x, filters=F, r=r, name_prefix="rrm2") # Bins: 60 -> 30
    x = residual_reduction_module_2d(x, filters=F, r=r, name_prefix="rrm3") # Bins: 30 -> 15
    
    x = residual_reduction_module_2d(x, filters=F, r=r, name_prefix="rrm4") # Bins: 15 -> 8
    
    
    x = layers.Flatten(name="flatten_features")(x)
    x = layers.Dropout(0.25, name="dropout_features")(x)
    
    common_feat = layers.Dense(128, activation='relu', name="dense_shared")(x)
    
    # Multi-Head Output
    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)
    
    combined_output = layers.Concatenate(axis=1, name="combined_output")([coords_output, mask_output])
    
    return models.Model(inputs=inputs, outputs=combined_output, name="EEAI_Net_V2_RRM")

In [ ]:
# ARCHITETTURA EEAI-NET V2 - DEPTHWISE SEPARABLE CONV BEST

def squeeze_excite_block_2d(x, filters, r=8):
    """Meccanismo di Attenzione spaziale basato su Squeeze-and-Excitation"""
    # Squeeze
    se = layers.GlobalAveragePooling2D()(x)
    # Excitation
    se = layers.Dense(max(1, filters // r), activation='relu', use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    # Reshape 
    se = layers.Reshape((1, 1, filters))(se)
    return layers.Multiply()([x, se])

def residual_reduction_module_2d_mobile(x, filters, r=8, name_prefix=""):
    # Depthwise Separable invece di Conv2D standard
    res = layers.DepthwiseConv2D(kernel_size=(1, 3), padding='same', use_bias=False, name=f"{name_prefix}_res_dw")(x)
    res = layers.BatchNormalization(name=f"{name_prefix}_res_bn1")(res)
    res = layers.ReLU(name=f"{name_prefix}_res_relu1")(res)
    res = layers.Conv2D(filters, kernel_size=(1, 1), padding='same', activation='relu', name=f"{name_prefix}_res_pw")(res)
    
    res = squeeze_excite_block_2d(res, filters, r=r)
    res = layers.Add(name=f"{name_prefix}_res_add")([res, x])

    red1 = layers.DepthwiseConv2D(kernel_size=(1, 3), strides=(1, 2), padding='same', use_bias=False, name=f"{name_prefix}_red_dw")(res)
    red1 = layers.BatchNormalization(name=f"{name_prefix}_red_bn2")(red1)
    red1 = layers.ReLU(name=f"{name_prefix}_red_relu2")(red1)
    red1 = layers.Conv2D(filters, kernel_size=(1, 1), padding='same', activation='relu', name=f"{name_prefix}_red_pw")(red1)
    
    # red2 resta una Conv2D standard 1x1 
    red2 = layers.Conv2D(filters, kernel_size=(1, 1), strides=(1, 2), padding='same', activation='relu', name=f"{name_prefix}_red_conv2")(res)
    
    out = layers.Add(name=f"{name_prefix}_red_add")([red1, red2])
    return out

def build_eeai_model_v2_rrm(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")
    
    F = 64
    r = 8  # Reduction ratio per il blocco SE (come da paper)
    
    # NOISE
    x = layers.GaussianNoise(0.01, name="input_noise")(inputs)

    # Feature Extraction Iniziale 
    x = layers.Conv2D(F, kernel_size=(1, 5), padding='same', activation='relu', name="init_conv")(x)
    
    x = residual_reduction_module_2d_mobile(x, filters=F, r=r, name_prefix="rrm1") # Bins: 120 -> 60
    x = residual_reduction_module_2d_mobile(x, filters=F, r=r, name_prefix="rrm2") # Bins: 60 -> 30
    x = residual_reduction_module_2d_mobile(x, filters=F, r=r, name_prefix="rrm3") # Bins: 30 -> 15

    x = residual_reduction_module_2d_mobile(x, filters=F, r=r, name_prefix="rrm4") # Bins: 15 -> 8
    
    x = layers.Flatten(name="flatten_features")(x)
    x = layers.Dropout(0.35, name="dropout_features")(x)
    
    common_feat = layers.Dense(128, activation='relu', name="dense_shared")(x)
    
    # Multi-Head Output 
    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)
    
    combined_output = layers.Concatenate(axis=1, name="combined_output")([coords_output, mask_output])
    
    return models.Model(inputs=inputs, outputs=combined_output, name="EEAI_Net_V2_RRM")

In [ ]:
import re
from pathlib import Path

summary_lines = []
model_rrm.summary(print_fn=lambda line: summary_lines.append(line))

layer_pattern = re.compile(r"(?P<layer>.+?)\s+\((?P<type>.+?)\)\s+(?P<output_shape>\[[^\]]+\]|\S+)\s+(?P<params>[0-9,]+)")
rows = []
for line in summary_lines:
    match = layer_pattern.match(line)
    if match:
        rows.append({
            "layer_name": match.group("layer").strip(),
            "layer_type": match.group("type").strip(),
            "output_shape": match.group("output_shape").strip(),
            "params": int(match.group("params").replace(",", ""))
        })

import pandas as pd
raw_df = pd.DataFrame({"summary_line": summary_lines})
parsed_df = pd.DataFrame(rows)
sheet_path = Path("/Users/davidepellegrino/Desktop/POLIMI/EEAI/Project/ProgettoEEAI/uwb-person-localization-tinyml/other_files/model_rrm_summary.xlsx")
sheet_path.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(sheet_path, engine="openpyxl") as writer:
    parsed_df.to_excel(writer, sheet_name="layers", index=False)
    raw_df.to_excel(writer, sheet_name="raw_summary", index=False)
    summary_info = [line for line in summary_lines if any(key in line for key in ["Total params:", "Trainable params:", "Non-trainable params:"])]
    if summary_info:
        info_df = pd.DataFrame({"info": summary_info})
        info_df.to_excel(writer, sheet_name="summary_info", index=False)

print(f"Saved model summary to {sheet_path}")

OSError: Cannot save file into a non-existent directory: 'other_files'

In [ ]:
import re
import pandas as pd
from pathlib import Path

sheet_path = Path("/Users/davidepellegrino/Desktop/POLIMI/EEAI/Project/ProgettoEEAI/uwb-person-localization-tinyml/other_files/model_rrm_summary.xlsx")
sheet_path.parent.mkdir(parents=True, exist_ok=True)

if 'summary_lines' not in globals():
    summary_lines = []
    model_rrm.summary(print_fn=lambda line: summary_lines.append(line))

layer_pattern = re.compile(r"(?P<layer>.+?)\s+\((?P<type>.+?)\)\s+(?P<output_shape>\[[^\]]+\]|\S+)\s+(?P<params>[0-9,]+)")
rows = []
for line in summary_lines:
    match = layer_pattern.match(line)
    if match:
        rows.append({
            'layer_name': match.group('layer').strip(),
            'layer_type': match.group('type').strip(),
            'output_shape': match.group('output_shape').strip(),
            'params': int(match.group('params').replace(',', ''))
        })

raw_df = pd.DataFrame({'summary_line': summary_lines})
parsed_df = pd.DataFrame(rows)

with pd.ExcelWriter(sheet_path, engine='openpyxl') as writer:
    parsed_df.to_excel(writer, sheet_name='layers', index=False)
    raw_df.to_excel(writer, sheet_name='raw_summary', index=False)
    summary_info = [line for line in summary_lines if any(key in line for key in ['Total params:', 'Trainable params:', 'Non-trainable params:'])]
    if summary_info:
        info_df = pd.DataFrame({'info': summary_info})
        info_df.to_excel(writer, sheet_name='summary_info', index=False)

print(f"Saved model summary to {sheet_path}")

In [ ]:
# TRAINING MODEL V2 (RRM) BEST


print("\n--- PREPARAZIONE TF.DATA PIPELINE ---")

train_dataset = tf.data.Dataset.from_tensor_slices((X_train, Y_train))
train_dataset = train_dataset.shuffle(buffer_size=5000).batch(32).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val, Y_val))
val_dataset = val_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

model_rrm = build_eeai_model_v2_rrm()

model_rrm.compile(
    optimizer='adam',
    loss=hungarian_total_loss, 
    metrics=[hungarian_rmse_metres, hungarian_mask_acc]
)

checkpoint_rrm = ModelCheckpoint("norm+depth+noise_model_toscano.keras", monitor="val_loss", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6, verbose=1)

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

EPOCHS = 100
embedded_summary(model_rrm, input_shape=(1, 120, 18), is_int8=False)

print("\n--- INIZIO ADDESTRAMENTO MODELLO RRM ---")
history_rrm = model_rrm.fit(
    train_dataset,
    validation_data= val_dataset,
    # batch_size=32,
    # shuffle=True,
    epochs=EPOCHS,
    callbacks=[checkpoint_rrm, reduce_lr, early_stop], 
    verbose=1
)
print("--- ADDESTRAMENTO COMPLETATO ---")

In [ ]:
# TRAINING MODEL V2 (RRM) 

print("\n--- PREPARAZIONE TF.DATA PIPELINE ---")
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, Y_train))
train_dataset = train_dataset.shuffle(buffer_size=5000).batch(32).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val, Y_val))
val_dataset = val_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

model_rrm = build_eeai_model_v2_rrm()

model_rrm.compile(
    optimizer='adam',
    loss=hungarian_total_loss, 
    metrics=[hungarian_rmse_metres, hungarian_mask_acc]
)

checkpoint_rrm = ModelCheckpoint("norm+depth+noise_split2_model_toscano.keras", monitor="val_loss", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6, verbose=1)

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

EPOCHS = 100
embedded_summary(model_rrm, input_shape=(1, 120, 18), is_int8=False)

print("\n--- INIZIO ADDESTRAMENTO MODELLO RRM SPLIT DIVERSO---")
history_rrm = model_rrm.fit(
    train_dataset,
    validation_data= val_dataset,
    # batch_size=32,
    # shuffle=True,
    epochs=EPOCHS,
    callbacks=[checkpoint_rrm, reduce_lr, early_stop], 
    verbose=1
)
print("--- ADDESTRAMENTO COMPLETATO ---")

In [ ]:
model_rrm = tf.keras.models.load_model(
    "best_model_toscano.keras", 
    custom_objects={
        'hungarian_total_loss': hungarian_total_loss,
        'hungarian_rmse_metres': hungarian_rmse_metres,
        'hungarian_mask_acc': hungarian_mask_acc
    }
)


optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5)
model_rrm.compile(
    optimizer=optimizer,
    loss=hungarian_total_loss, 
    metrics=[hungarian_rmse_metres, hungarian_mask_acc]
)

checkpoint_ft = tf.keras.callbacks.ModelCheckpoint("finetuned_model.keras", monitor="val_loss", save_best_only=True, verbose=1)
early_stop_ft = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

print("\n--- INIZIO FINE-TUNING CON RADAR DROPOUT ---")
history_ft = model_rrm.fit(
    train_dataset,               
    validation_data=val_dataset,
    epochs=30,                   
    callbacks=[checkpoint_ft, early_stop_ft], 
    verbose=1
)

In [ ]:
# VISUALIZER

FILE_TARGET = "/home/marco/Desktop/test_project_edge_ai/other_files/dataset/data/window_000021.npz"
KERAS_MODEL_PATH = "/home/marco/Desktop/test_project_edge_ai/other_files/best_model_toscano.keras"
TFLITE_MODEL_PATH = "/home/marco/Desktop/test_project_edge_ai/submission/model.tflite"

ALPHA = 0.02
GLOBAL_MAX = 203.93

if not os.path.exists(FILE_TARGET):
    print(f"ERRORE: Non trovo il file {FILE_TARGET}")
else:
    print("1️⃣ Caricamento Dati e Pre-processing (EMA + Normalizzazione)...")
    data = np.load(FILE_TARGET)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] 
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2).reshape(T, 1, 120, 18)
    decluttered = np.zeros_like(mag)
    bg = np.copy(mag[0])
    
    for t in range(T):
        bg = ALPHA * mag[t] + (1 - ALPHA) * bg
        decluttered[t] = np.abs(mag[t] - bg)
        
    # NORMALIZZAZIONE
    normalized_data = np.clip(decluttered, 0, GLOBAL_MAX) / GLOBAL_MAX

 
    print("Inferenza Modello Keras (Float32)...")
    model_keras = tf.keras.models.load_model(KERAS_MODEL_PATH, compile=False)
    preds_keras = model_keras.predict(normalized_data, verbose=0)
    
    k_coords = preds_keras[:, :8].reshape(T, 4, 2)
    k_mask = preds_keras[:, 8:]

    # ==============================================================================
    # INFERENZA TFLITE (INT 8)
    # ==============================================================================
    print("Inferenza Modello TFLite (INT8)... (Attendi qualche secondo)")
    interpreter = tf.lite.Interpreter(
        model_path=TFLITE_MODEL_PATH,
    )
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    in_scale, in_zp = input_details['quantization']
    out_scale, out_zp = output_details['quantization']

    tfl_coords = np.zeros((T, 4, 2))
    tfl_mask = np.zeros((T, 4))

    for t in range(T):
        input_float = np.expand_dims(normalized_data[t].astype(np.float32), axis=0)
        
        # Quantizza
        input_quant = np.round(input_float / in_scale) + in_zp
        input_quant = np.clip(input_quant, -128, 127).astype(np.int8)
        
        interpreter.set_tensor(input_details['index'], input_quant)
        interpreter.invoke()
        
        # Dequantizza
        preds_quant = interpreter.get_tensor(output_details['index'])[0]
        preds_float = (preds_quant.astype(np.float32) - out_zp) * out_scale
        
        tfl_coords[t] = preds_float[:8].reshape(4, 2)
        tfl_mask[t] = preds_float[8:]

    print("✅ Dati pronti! Inizializzazione Interfaccia Grafica...")

    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(6, 8)) 
            ax.set_xlim(-0.5, 5.3); ax.set_ylim(-0.5, 7.7)
            ax.grid(True, linestyle=':', alpha=0.6)
            
            ax.set_title(f"Confronto Keras vs TFLite | Window: 21 | Frame: {frame_idx}/{T-1}", fontsize=12, fontweight='bold')

            stanza = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=3, edgecolor='navy', facecolor='whitesmoke')
            ax.add_patch(stanza)

            for i in range(4):
                if gt_mask[frame_idx, i] > 0.5:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=250, edgecolors='black', marker='o', alpha=0.4, label='Ground Truth' if i==0 else "")
                    ax.text(rx, ry + 0.25, f"GT_{i+1}", color='darkgreen', fontweight='bold', ha='center')

            for i in range(4):
                conf_k = float(k_mask[frame_idx, i])
                if conf_k >= soglia:
                    px, py = k_coords[frame_idx, i]
                    ax.scatter(px, py, c='blue', s=120, marker='s', edgecolors='darkblue', alpha=0.8, label='Pred Keras (Float32)' if i==0 else "")
                    ax.text(px, py - 0.25, f"K:{conf_k*100:.0f}%", color='blue', fontsize=9, ha='center', fontweight='bold')

            for i in range(4):
                conf_tfl = float(tfl_mask[frame_idx, i])
                if conf_tfl >= soglia:
                    px, py = tfl_coords[frame_idx, i]
                    ax.scatter(px, py, c='red', s=100, marker='X', edgecolors='darkred', alpha=0.9, label='Pred TFLite (INT8)' if i==0 else "")
                    ax.text(px, py - 0.45, f"T:{conf_tfl*100:.0f}%", color='red', fontsize=9, ha='center', fontweight='bold')

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if by_label:
                ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True, shadow=True, fontsize=10)

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    slider_frame = widgets.IntSlider(value=500, min=10, max=T-1, step=1, description='Frame:', layout=widgets.Layout(width='400px'))
    slider_soglia = widgets.FloatSlider(value=0.85, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    controls = widgets.VBox([slider_frame, slider_soglia])
    controls.layout.margin = '20px 20px 20px 0px' 
    ui = widgets.HBox([controls, out])
    
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)